# RDD (Resilient Distributed Database)

- Resilient -> Fauly tolerent (using - DAG and immutability of RDD)
- Distributed on different executors/- workers
- Immutable - creates new copy for - each trasformation
- Used for unstrcutered data, No - schema
- Type Safe: Give compile time error - for data type errors
- Not optimized by spark. Developer - need to optimize. (Full control on data by Developer)

## Create RDD and perform simple iterative operation

In [0]:
# %python
# Setup
sc = spark.sparkContext

# Simple RDD from an in-memory list
nums_rdd = sc.parallelize(range(1, 11))   # 1..10

# transformations and actions
squares = nums_rdd.map(lambda x: x * x)
even_squares = squares.filter(lambda x: x % 2 == 0)

print("Squares:", squares.collect())
print("Even squares:", even_squares.collect())
print("Count of even squares:", even_squares.count())

## Count Words

In [0]:
# %python
lines = [
  "Apache Spark makes big data processing easy",
  "RDD demo in Databricks",
  "Use RDD when you need low-level control; prefer DataFrame for SQL-style ops"
]

lines_rdd = sc.parallelize(lines)
words = lines_rdd.flatMap(lambda line: line.split())
word_counts = words.map(lambda w: (w.lower().strip(), 1)).reduceByKey(lambda a, b: a + b)
print("Word counts:", word_counts.collect())

## Join and ReduceByKey

In [0]:
# %python
names = sc.parallelize([("id1", "Alice"), ("id2", "Bob"), ("id3", "Cathy")])
scores = sc.parallelize([("id1", 95), ("id2", 80), ("id1", 88), ("id4", 70)])

# reduceByKey: total score per id
total_scores = scores.reduceByKey(lambda a, b: a + b)
print("Total scores:", total_scores.collect())

# join names with total_scores (inner join)
joined = names.join(total_scores)
print("Joined (id -> (name, total_score)):", joined.collect())

## mapPartitions example

In [0]:
# %python
def partition_sum(iterator):
    total = 0
    for v in iterator:
        total += v
    yield total

big_rdd = sc.parallelize(range(1, 1001), 8)   # 8 partitions
per_partition_sums = big_rdd.mapPartitions(partition_sum).collect()
print("Per-partition sums:", per_partition_sums)
print("Total sum (verify):", sum(per_partition_sums))

## persist/cache vs checkpoint + lineage

In [0]:
# %python
# set checkpoint directory on DBFS (Databricks)
sc.setCheckpointDir("dbfs:/tmp/rdd_checkpoints_demo")

rdd = sc.parallelize(range(1, 100001)).map(lambda x: x * 2).filter(lambda x: x % 3 == 0)

# show lineage before caching/checkpoint
print("Lineage before actions:\n", rdd.toDebugString())

# cache and an action
rdd_cached = rdd.persist()   # or .cache()
count_before = rdd_cached.count()
print("Count after cache & action:", count_before)
print("Lineage after cache (still logical):\n", rdd_cached.toDebugString())

# checkpoint the RDD (materializes on next action)
rdd_cached.checkpoint()
rdd_cached.count()   # triggers checkpoint
print("Lineage after checkpoint (should be truncated):\n", rdd_cached.toDebugString())

# cleanup
rdd_cached.unpersist()

## aggregate example (compute mean)

In [0]:
# %python
r = sc.parallelize([1,2,3,4,5,6,7,8,9,10])

# aggregate: (sum, count)
zero = (0, 0)
seqOp = lambda acc, v: (acc[0] + v, acc[1] + 1)
combOp = lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1])

sum_count = r.aggregate(zero, seqOp, combOp)
mean = sum_count[0] / float(sum_count[1])
print("Mean:", mean)

## Convert RDD -> DataFrame and back

In [0]:
# %python
pairs = sc.parallelize([(1, "Alice"), (2, "Bob"), (3, "Cathy")])
df = pairs.toDF(["id", "name"])
display(df)   # Databricks display

# back to RDD
rdd_from_df = df.rdd
print("RDD from DF:", rdd_from_df.collect())

## 